# Омни-ассистент: экскурсия голосом персонажа

[![Открыть в Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samtakoy/llm-engineer-OMNI-assistant-dz/blob/main/notebooks/omni_gradio.ipynb)

Тетрадь идёт в двух средах.

**Локально.** Ядро - `.venv` проекта: пакет `assistant` ставится туда через
`uv sync`. Модели раздаёт ollama, поднятая на своей машине.

**В Colab.** Нужна среда с видеокартой: Среда выполнения → Сменить среду
выполнения → T4 GPU. Клетки сами склонируют репозиторий, поставят пакет и
поднимут ollama.

Порядок работы в интерфейсе: загрузить фотографию персонажа, задать вопрос
текстом или голосом, нажать кнопку. Описание персонажа, текст лекции и
проигрыватели с озвучкой появляются по мере готовности.

In [ ]:
import sys

IS_COLAB = "google.colab" in sys.modules

print(f"среда: {'colab' if IS_COLAB else 'локальная'}")
print(f"питон: {sys.version.split()[0]}")

In [ ]:
if IS_COLAB:
    !nvidia-smi
    !git clone https://github.com/samtakoy/llm-engineer-OMNI-assistant-dz.git /content/omni
    # Ставим правкой на месте: PROJECT_ROOT в variables.py считается от файла
    # пакета, и при обычной установке каталоги проекта уехали бы в site-packages.
    !pip install -e /content/omni
else:
    print("локальная среда: пакет стоит в .venv, установка пропущена")

In [ ]:
import json
import subprocess
import time
import urllib.error
import urllib.request

# Имена моделей отсюда уезжают в переменные окружения следующей клетки.
# Бюджет видеокарты T4: 4b в четырёхбитном кванте около 2.5 ГБ, зрение 4b около
# 3 ГБ, распознавание речи около 1.5 ГБ, синтез около 0.2 ГБ.
TEXT_MODEL = "qwen3.5:4b"
VISION_MODEL_NAME = "qwen3-vl:4b"

OLLAMA_URL = "http://127.0.0.1:11434"


def wait_for_ollama(seconds: int) -> bool:
    """
    Ждёт, пока сервер ollama начнёт отвечать.

    Аргументы:
        seconds: сколько ждать.

    Возвращает:
        True, если сервер ответил за отведённое время.
    """
    deadline = time.monotonic() + seconds

    while time.monotonic() < deadline:
        try:
            with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout = 2):
                return True
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            time.sleep(2)

    return False


def installed_models() -> set[str]:
    """
    Спрашивает у ollama список загруженных моделей.

    Возвращает:
        Имена моделей вместе с тегами.
    """
    with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout = 5) as answer:
        listing = json.load(answer)

    return {item["name"] for item in listing.get("models", [])}


if IS_COLAB:
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(
        ["ollama", "serve"],
        stdout = subprocess.DEVNULL,
        stderr = subprocess.DEVNULL,
    )

if not wait_for_ollama(seconds = 60):
    raise RuntimeError(f"ollama не отвечает на {OLLAMA_URL}: подними сервер и повтори клетку")

present = installed_models()
missing = [name for name in (TEXT_MODEL, VISION_MODEL_NAME) if name not in present]

if not missing:
    print(f"модели на месте: {TEXT_MODEL}, {VISION_MODEL_NAME}")
elif IS_COLAB:
    for name in missing:
        !ollama pull {name}
else:
    # Гигабайты на чужую машину без спроса не тянем.
    print("не хватает моделей, скачай и повтори клетку:")
    for name in missing:
        print(f"    ollama pull {name}")

In [ ]:
import os

# Выставляется до первого импорта assistant: variables.py читает окружение один
# раз, а load_dotenv уже выставленные переменные не перебивает.
# Имя поля с размышлением здесь не задаётся: его знает провайдер.
# Возврат на lm studio - значение "local" в первой строке.
os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = TEXT_MODEL
os.environ["VISION_PROVIDER"] = "ollama"
os.environ["VISION_MODEL"] = VISION_MODEL_NAME

In [ ]:
from assistant.webui import launch_app

# Публичная ссылка нужна только в colab: во встроенном фрейме микрофон
# работает не всегда.
launch_app(is_inline = False, is_shared = IS_COLAB)